# UC05 — Analítica de Exposición a Catástrofes

Análisis geoespacial de exposición CAT con PML, concentración y escenarios de reaseguro.
**Valor de negocio**: Visibilidad en tiempo real de la exposición catastrófica para decisiones de reaseguro.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS EXPOSICION_CAT_DB;
USE SCHEMA EXPOSICION_CAT_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Cartera de Pólizas Geolocalizadas

In [ ]:
CREATE OR REPLACE TABLE polizas_geo AS
SELECT
    'POL' || LPAD(SEQ4(), 5, '0') AS poliza_id,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Hogar' WHEN 1 THEN 'Comercio' ELSE 'Industrial' END AS tipo,
    CASE MOD(SEQ4(), 10)
        WHEN 0 THEN 'Madrid' WHEN 1 THEN 'Barcelona' WHEN 2 THEN 'Valencia' WHEN 3 THEN 'Sevilla'
        WHEN 4 THEN 'Málaga' WHEN 5 THEN 'Murcia' WHEN 6 THEN 'Bilbao'
        WHEN 7 THEN 'Zaragoza' WHEN 8 THEN 'Alicante' ELSE 'Granada'
    END AS provincia,
    UNIFORM(50000, 2000000, RANDOM())::DECIMAL(12,2) AS suma_asegurada,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Hormigon' WHEN 1 THEN 'Ladrillo' ELSE 'Madera' END AS tipo_construccion,
    UNIFORM(1, 80, RANDOM()) AS antiguedad_inmueble,
    UNIFORM(1, 10, RANDOM()) AS num_plantas
FROM TABLE(GENERATOR(ROWCOUNT => 2000));

SELECT tipo, provincia, COUNT(*) AS polizas, SUM(suma_asegurada)::DECIMAL(15,2) AS exposicion_total
FROM polizas_geo GROUP BY tipo, provincia ORDER BY exposicion_total DESC LIMIT 15;

---

## Paso 3: Generar Zonas de Riesgo

In [ ]:
CREATE OR REPLACE TABLE zonas_riesgo AS
SELECT
    'ZONA' || LPAD(SEQ4(), 3, '0') AS zona_id,
    CASE MOD(SEQ4(), 10)
        WHEN 0 THEN 'Madrid' WHEN 1 THEN 'Barcelona' WHEN 2 THEN 'Valencia' WHEN 3 THEN 'Sevilla'
        WHEN 4 THEN 'Málaga' WHEN 5 THEN 'Murcia' WHEN 6 THEN 'Bilbao'
        WHEN 7 THEN 'Zaragoza' WHEN 8 THEN 'Alicante' ELSE 'Granada'
    END AS provincia,
    CASE MOD(SEQ4(), 4) WHEN 0 THEN 'Inundacion' WHEN 1 THEN 'Terremoto' WHEN 2 THEN 'Granizo' ELSE 'Incendio' END AS tipo_peligro,
    UNIFORM(1, 5, RANDOM()) AS nivel_peligro,
    UNIFORM(1, 15, RANDOM()) AS frecuencia_10a,
    UNIFORM(100000, 5000000, RANDOM())::DECIMAL(12,2) AS perdida_media_evento
FROM TABLE(GENERATOR(ROWCOUNT => 50));

SELECT * FROM zonas_riesgo ORDER BY nivel_peligro DESC LIMIT 10;

---

## Paso 4: Calcular Exposición Agregada por Zona y Tramo de Reaseguro

In [ ]:
CREATE OR REPLACE VIEW exposicion_agregada AS
SELECT
    z.zona_id, z.provincia, z.tipo_peligro, z.nivel_peligro,
    COUNT(p.poliza_id) AS num_polizas,
    SUM(p.suma_asegurada)::DECIMAL(15,2) AS suma_total,
    (SUM(p.suma_asegurada) * z.nivel_peligro / 10)::DECIMAL(15,2) AS pml_estimado,
    LEAST(SUM(p.suma_asegurada) * z.nivel_peligro / 10, 5000000)::DECIMAL(15,2) AS retencion,
    GREATEST(LEAST(SUM(p.suma_asegurada) * z.nivel_peligro / 10, 20000000) - 5000000, 0)::DECIMAL(15,2) AS xl_1,
    GREATEST(SUM(p.suma_asegurada) * z.nivel_peligro / 10 - 20000000, 0)::DECIMAL(15,2) AS xl_2
FROM zonas_riesgo z JOIN polizas_geo p ON z.provincia = p.provincia
GROUP BY z.zona_id, z.provincia, z.tipo_peligro, z.nivel_peligro;

SELECT * FROM exposicion_agregada ORDER BY pml_estimado DESC LIMIT 15;

---

## Paso 5: Análisis de Concentración

In [ ]:
SELECT provincia, COUNT(*) AS zonas, SUM(num_polizas) AS polizas, SUM(pml_estimado)::DECIMAL(15,2) AS pml_total,
    ROUND(SUM(pml_estimado)/NULLIF(SUM(num_polizas),0),2) AS pml_por_poliza
FROM exposicion_agregada GROUP BY provincia ORDER BY pml_total DESC;

---

## Paso 6: Modelización de Escenarios CAT

In [ ]:
CREATE OR REPLACE TABLE escenarios AS
SELECT 'Inundación severa Madrid' AS escenario, 'Madrid' AS provincia, 'Inundacion' AS peligro, 0.35 AS factor_afectacion
UNION ALL SELECT 'Terremoto Murcia','Murcia','Terremoto',0.25
UNION ALL SELECT 'Granizada Barcelona','Barcelona','Granizo',0.40
UNION ALL SELECT 'Incendio forestal Valencia','Valencia','Incendio',0.20
UNION ALL SELECT 'Inundación Sevilla','Sevilla','Inundacion',0.30;

SELECT e.escenario,
    COUNT(p.poliza_id) AS polizas_afectadas,
    (SUM(p.suma_asegurada) * e.factor_afectacion)::DECIMAL(15,2) AS perdida_bruta,
    LEAST(SUM(p.suma_asegurada) * e.factor_afectacion, 5000000)::DECIMAL(15,2) AS retencion_neta,
    GREATEST(SUM(p.suma_asegurada) * e.factor_afectacion - 5000000, 0)::DECIMAL(15,2) AS cesion_reaseguro
FROM escenarios e JOIN polizas_geo p ON e.provincia = p.provincia
GROUP BY e.escenario, e.factor_afectacion
ORDER BY perdida_bruta DESC;

---

## Paso 7: Dashboard de Exposición CAT

In [ ]:
import pandas as pd

session = get_active_session()
st.title('Analítica de Exposición a Catástrofes')

exp = session.sql('SELECT * FROM exposicion_agregada').to_pandas()
esc = session.sql('SELECT e.escenario, COUNT(p.poliza_id) AS polizas, (SUM(p.suma_asegurada)*e.factor_afectacion)::DECIMAL(15,2) AS perdida_bruta FROM escenarios e JOIN polizas_geo p ON e.provincia=p.provincia GROUP BY e.escenario,e.factor_afectacion ORDER BY perdida_bruta DESC').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Exposición Total', f"{exp["SUMA_TOTAL"].sum():,.0f}€")
with c2: st.metric('PML Máximo', f"{exp["PML_ESTIMADO"].max():,.0f}€")
with c3: st.metric('Zonas Críticas', len(exp[exp['NIVEL_PELIGRO']>=4]))

st.markdown('---')
st.subheader('Exposición por Tipo de Peligro')
peligro = exp.groupby('TIPO_PELIGRO')['PML_ESTIMADO'].sum()
st.bar_chart(pd.DataFrame({'PML': peligro.values}, index=peligro.index))

st.markdown('---')
st.subheader('Escenarios CAT')
st.dataframe(esc, use_container_width=True)

st.markdown('---')
st.subheader('Top Zonas por PML')
st.dataframe(exp.nlargest(15,'PML_ESTIMADO')[['ZONA_ID','PROVINCIA','TIPO_PELIGRO','NUM_POLIZAS','PML_ESTIMADO','RETENCION','XL_1','XL_2']], use_container_width=True)
st.caption('Desarrollado con Snowflake | Datos: Sintéticos')

---

## Paso 8: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS EXPOSICION_CAT_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;